# 02 — Input bit-depth sweep (PyTorch)

Sweeps input quantization in [0,2,4,8] with fixed network precision.

In [ ]:
from pathlib import Path
import sys, os

# ---- Path setup (adjust if your repo layout differs) ----
PROJECT_ROOT = Path("..").resolve()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

In [ ]:
import pandas as pd

from config import ExperimentConfig, with_overrides
from runner import run_experiment
from utils import load_runs, flatten_runs
from plots import (
    plot_metric_vs_input_bits,
    plot_delta_from_baseline,
    plot_tradeoff_with_pareto,
)

pd.set_option("display.max_columns", 200)


In [11]:

base = ExperimentConfig(
    backend="pytorch",
    device="cuda",
    batch_size=1,
    model_precision="fp32",      
    input_quant_bits=0,        
    seed=42,
    num_eval_batches=100,
)

in_bits_list = [8, 4, 2, 1] 

cfgs = [with_overrides(base, input_quant_bits=b) for b in in_bits_list]

# cfgs += [with_overrides(base, model_precision="fp16", input_quant_bits=b) for b in in_bits_list]

In [12]:
records = []
for cfg in cfgs:
    payload, _ = run_experiment(cfg, split="val", save_results_flag=True)
    records.append(payload)

for r in records:
    print(r["run_id"], r["status"], r["results"].get("top1_acc"))

Evaluating on 100 batches...
  Batch [10/100] Top-1: 80.00% | Top-5: 100.00% | Infer: 9.66 ms/batch
  Batch [20/100] Top-1: 85.00% | Top-5: 95.00% | Infer: 7.38 ms/batch
  Batch [30/100] Top-1: 90.00% | Top-5: 96.67% | Infer: 6.00 ms/batch
  Batch [40/100] Top-1: 85.00% | Top-5: 95.00% | Infer: 5.07 ms/batch
  Batch [50/100] Top-1: 88.00% | Top-5: 96.00% | Infer: 4.61 ms/batch
  Batch [60/100] Top-1: 85.00% | Top-5: 95.00% | Infer: 4.33 ms/batch
  Batch [70/100] Top-1: 84.29% | Top-5: 95.71% | Infer: 4.17 ms/batch
  Batch [80/100] Top-1: 85.00% | Top-5: 95.00% | Infer: 3.98 ms/batch
  Batch [90/100] Top-1: 84.44% | Top-5: 95.56% | Infer: 3.90 ms/batch
  Batch [100/100] Top-1: 85.00% | Top-5: 96.00% | Infer: 3.83 ms/batch
[saved] runs/resnet18_pytorch_fp32_in8b_cuda_bs1/result.json
Evaluating on 100 batches...
  Batch [10/100] Top-1: 70.00% | Top-5: 100.00% | Infer: 3.73 ms/batch
  Batch [20/100] Top-1: 80.00% | Top-5: 95.00% | Infer: 3.35 ms/batch
  Batch [30/100] Top-1: 86.67% | Top-5

In [ ]:
runs = load_runs("./runs", status="ok")
rows = flatten_runs(runs)
df = pd.DataFrame(rows)

# Filter only what this notebook is responsible for
df_sweep = df[
    (df["cfg.backend"] == "pytorch")
    & (df["cfg.device"] == "cuda")
    & (df["cfg.model_precision"].isin(["fp32"]))   # include "fp16" if you enabled it
    & (df["cfg.input_quant_bits"].isin([0, 8, 4, 2, 1]))
].copy()

df_sweep[[
    "run_id",
    "cfg.model_precision",
    "cfg.input_quant_bits",
    "res.top1_acc",
    "res.top5_acc",
    "res.infer_ms_avg",
    "res.throughput_infer_sps",
    "res.total_samples",
]].sort_values(["cfg.model_precision", "cfg.input_quant_bits"])

In [ ]:
rows_sweep = df_sweep.to_dict(orient="records")

plot_metric_vs_input_bits(
    rows_sweep,
    metric_key="res.top1_acc",
    title="PyTorch: Top-1 vs input bit-depth",
    connect_points=False,
)

plot_delta_from_baseline(
    rows_sweep,
    baseline_selector={
        "cfg.backend": "pytorch",
        "cfg.model_precision": "fp32",
        "cfg.input_quant_bits": 0,
        "cfg.device": "cuda",
    },
    title="Input quantization sweep vs FP32 baseline",
)

plot_tradeoff_with_pareto(
    rows_sweep,
    x_key="res.infer_ms_avg",
    y_key="res.top1_acc",
    title="Input quantization: Accuracy–Latency tradeoff (Pareto)",
)